# 02 - Silver

Lê o bronze inteiro e gera uma tabela por recurso com schema fixo, textos e datas padronizados e uma linha por registro.
O que quebra regra vai para silver_rejeitados com o motivo.

In [ ]:
PASTA_ARQUIVOS_SPARK = "Files"
FORMATO_TABELA = "delta"

In [ ]:
from functools import reduce

from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F
from pyspark.sql.types import (ArrayType, BooleanType, DoubleType, IntegerType,
                               LongType, StringType, StructField, StructType)

spark.conf.set("spark.sql.session.timeZone", "UTC")
BRONZE = f"{PASTA_ARQUIVOS_SPARK}/bronze"

PRIORIDADES = ["Baixa", "Média", "Alta", "Crítica"]
ETAPAS_TAREFA = ["Backlog", "A fazer", "Fazendo", "Impedido", "Em revisão", "Concluído", "Arquivada"]
SITUACOES_PROJETO = ["Ativo", "Concluído", "Arquivado"]
FAROIS = ["Verde", "Amarelo", "Vermelho", "Encerrado"]


def campos(*definicoes: tuple[str, object]) -> StructType:
    return StructType([StructField(nome, tipo, True) for nome, tipo in definicoes])


S, I, D = StringType(), IntegerType(), DoubleType()
CONTRATOS = {
    "portfolios": campos(("id", I), ("nome", S), ("sigla", S), ("dono_id", S), ("etapas", ArrayType(S))),
    "usuarios": campos(("id", S), ("nome", S), ("email", S), ("portfolio_id", I), ("cargo", S),
                       ("data_admissao", S), ("ativo", BooleanType())),
    "projetos": campos(("id", S), ("portfolio_id", I), ("nome", S), ("etapa", S), ("situacao", S),
                       ("etiquetas", ArrayType(S)), ("farol", S), ("origem", S), ("origem_id", S),
                       ("gerente_id", S), ("prioridade", S), ("inicio", S), ("limite", S), ("concluido_em", S),
                       ("horas_orcadas", I), ("orcamento", D), ("criado_em", S), ("atualizado_em", S)),
    "tarefas": campos(("id", S), ("projeto_id", S), ("titulo", S), ("etapa", S), ("situacao", S),
                      ("etiquetas", ArrayType(S)), ("responsavel_id", S), ("prioridade", S), ("tipo", S),
                      ("estimativa_horas", D), ("horas_apontadas", D), ("criada_em", S), ("limite", S),
                      ("concluida_em", S), ("atualizado_em", S)),
    "movimentacoes": campos(("id", S), ("tarefa_id", S), ("etapa_anterior", S), ("etapa_nova", S),
                            ("usuario_id", S), ("ocorrido_em", S)),
    "meta": campos(("versao", S), ("data_referencia", S), ("semente", LongType())),
}


def ler_bronze(recurso: str) -> DataFrame:
    caminho = f"{BRONZE}/{recurso}/*/*.json"
    if recurso == "meta":
        return spark.read.schema(CONTRATOS["meta"]).option("multiLine", True).json(caminho) \
            .withColumn("_arquivo", F.col("_metadata.file_path"))
    envelope = campos(("dados", ArrayType(CONTRATOS[recurso])), ("_execucao", S))
    return (spark.read.schema(envelope).option("multiLine", True).json(caminho)
            .select(F.explode("dados").alias("r"), "_execucao")
            .select("r.*", "_execucao"))


def ultima_foto(df: DataFrame) -> DataFrame:
    ultima = df.agg(F.max("_execucao")).first()[0]
    return df.filter(F.col("_execucao") == ultima)


def mais_recente(df: DataFrame, ordem: list) -> DataFrame:
    janela = Window.partitionBy("id").orderBy(*ordem)
    return df.withColumn("_n", F.row_number().over(janela)).filter("_n = 1").drop("_n", "_execucao")


def texto(coluna: str):
    limpo = F.regexp_replace(F.trim(F.col(coluna)), r"\s+", " ")
    return F.when(limpo == "", None).otherwise(limpo)


def instante(coluna: str):
    return F.to_timestamp(F.col(coluna))


def prioridade(coluna: str):
    p = F.initcap(F.lower(texto(coluna)))
    return F.when(p.isin(PRIORIDADES), p).otherwise(F.lit("Não informada"))

In [ ]:
portfolios = mais_recente(ultima_foto(ler_bronze("portfolios")), [F.desc("_execucao")]).select(
    "id", texto("nome").alias("nome"), F.upper(texto("sigla")).alias("sigla"), "dono_id", "etapas")

usuarios = mais_recente(ultima_foto(ler_bronze("usuarios")), [F.desc("_execucao")]).select(
    "id", texto("nome").alias("nome"), F.lower(texto("email")).alias("email"), "portfolio_id",
    texto("cargo").alias("cargo"), F.to_date("data_admissao").alias("data_admissao"),
    F.coalesce("ativo", F.lit(True)).alias("ativo"))

projetos = mais_recente(ler_bronze("projetos"), [F.desc(instante("atualizado_em")), F.desc("_execucao")]).select(
    "id", "portfolio_id", texto("nome").alias("nome"), texto("etapa").alias("etapa"),
    texto("situacao").alias("situacao"), F.coalesce("etiquetas", F.array().cast("array<string>")).alias("etiquetas"),
    F.initcap(texto("farol")).alias("farol"), texto("origem").alias("origem"), "origem_id", "gerente_id",
    prioridade("prioridade").alias("prioridade"), F.to_date("inicio").alias("inicio"),
    F.to_date("limite").alias("limite"), F.to_date("concluido_em").alias("concluido_em"), "horas_orcadas",
    "orcamento", instante("criado_em").alias("criado_em"), instante("atualizado_em").alias("atualizado_em"))

tarefas = mais_recente(ler_bronze("tarefas"), [F.desc(instante("atualizado_em")), F.desc("_execucao")]).select(
    "id", "projeto_id", texto("titulo").alias("titulo"), texto("tipo").alias("tipo"),
    prioridade("prioridade").alias("prioridade"), "responsavel_id",
    "estimativa_horas", "horas_apontadas", texto("etapa").alias("etapa"), texto("situacao").alias("situacao"),
    F.coalesce("etiquetas", F.array().cast("array<string>")).alias("etiquetas"),
    instante("criada_em").alias("criada_em"), F.to_date("limite").alias("limite"),
    instante("concluida_em").alias("concluida_em"), instante("atualizado_em").alias("atualizado_em"))

movimentacoes = mais_recente(ler_bronze("movimentacoes"), [F.desc("_execucao")]).select(
    "id", "tarefa_id", texto("etapa_anterior").alias("etapa_anterior"),
    texto("etapa_nova").alias("etapa_nova"), "usuario_id", instante("ocorrido_em").alias("ocorrido_em"))

meta = (ler_bronze("meta").orderBy(F.desc("_arquivo")).limit(1)
        .select(F.to_date("data_referencia").alias("data_referencia"), "versao", "semente",
                F.current_timestamp().alias("processado_em")))

In [ ]:
ids = lambda df: df.select(F.col("id").alias("_ref"))
regras = {
    "projetos": [
        (~F.col("situacao").isin(SITUACOES_PROJETO) | F.col("situacao").isNull(), "situação de projeto desconhecida"),
        (~F.col("farol").isin(FAROIS), "farol desconhecido"),
        (F.col("limite") < F.col("inicio"), "limite antes do início"),
    ],
    "tarefas": [
        (~F.col("etapa").isin(ETAPAS_TAREFA) | F.col("etapa").isNull(), "etapa de tarefa desconhecida"),
        (F.col("estimativa_horas") < 0, "estimativa negativa"),
        (F.col("horas_apontadas") < 0, "horas negativas"),
    ],
    "movimentacoes": [
        (~F.col("etapa_nova").isin(ETAPAS_TAREFA) | F.col("etapa_nova").isNull(), "etapa desconhecida"),
    ],
}
chaves = [("usuarios", "portfolio_id", "portfolios"), ("projetos", "portfolio_id", "portfolios"),
          ("projetos", "gerente_id", "usuarios"), ("tarefas", "projeto_id", "projetos"),
          ("tarefas", "responsavel_id", "usuarios"), ("movimentacoes", "tarefa_id", "tarefas"),
          ("movimentacoes", "usuario_id", "usuarios")]

tabelas = {"portfolios": portfolios, "usuarios": usuarios, "projetos": projetos,
           "tarefas": tarefas, "movimentacoes": movimentacoes}
rejeitados = []


def separar(nome: str, df: DataFrame, condicao, motivo: str) -> DataFrame:
    ruins = df.filter(F.coalesce(condicao, F.lit(False)))
    rejeitados.append(ruins.select(F.lit(nome).alias("tabela"), F.col("id").cast("string").alias("id"),
                                   F.lit(motivo).alias("motivo"), F.to_json(F.struct("*")).alias("registro")))
    return df.filter(~F.coalesce(condicao, F.lit(False)))


for nome, lista in regras.items():
    for condicao, motivo in lista:
        tabelas[nome] = separar(nome, tabelas[nome], condicao, motivo)

for nome, coluna, referencia in chaves:
    df = tabelas[nome]
    validos = ids(tabelas[referencia])
    orfaos = df.join(validos, df[coluna] == validos["_ref"], "left_anti")
    rejeitados.append(orfaos.select(F.lit(nome).alias("tabela"), F.col("id").cast("string").alias("id"),
                                    F.lit(f"{coluna} sem correspondente").alias("motivo"),
                                    F.to_json(F.struct("*")).alias("registro")))
    tabelas[nome] = df.join(validos, df[coluna] == validos["_ref"], "left_semi")

rejeitados_df = reduce(DataFrame.unionByName, rejeitados).withColumn("processado_em", F.current_timestamp())

In [ ]:
def salvar(df: DataFrame, nome: str) -> int:
    (df.write.format(FORMATO_TABELA).mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable(nome))
    return spark.table(nome).count()


contagens = {f"silver_{n}": salvar(df, f"silver_{n}") for n, df in tabelas.items()}
contagens["silver_meta"] = salvar(meta, "silver_meta")
contagens["silver_rejeitados"] = salvar(rejeitados_df, "silver_rejeitados")

for nome, qtd in contagens.items():
    print(f"{nome:<20} {qtd:>6}")
if contagens["silver_rejeitados"]:
    display(spark.table("silver_rejeitados").groupBy("tabela", "motivo").count())